# Module 1: The "Invisible GPU" Trap (Debugging 0% Utilization)

Welcome! In this module, we will explore the classic MLOps incident: the **Invisible GPU Trap**.

**Section Goals:**
* Identify why model jobs run slowly even with GPU runtime enabled.
* Verify model and tensor placement.
* Fix the device placement to route workloads to CUDA.

### The Scenario

A Data Scientist approaches you complaining that their model training is taking hours. They show you that the Google Colab/cloud instance has a Tesla T4 GPU enabled.

You check the metrics and see **0% GPU Utilization** while the **CPU is at 100%**.

This happens because they allocated the GPU hardware but forgot to tell PyTorch to move the model and tensors onto it. The code is running entirely on the CPU!

### Visualizing the Trap

Here is the mismatch between CPU and GPU activity when device allocation is missing:

![Invisible GPU](images/invisible-gpu.svg)

### Step 1: Simulating the CPU Trap

Let's write a script to build a simple tensor but leave it on the CPU by mistake.

In [ ]:
import torch

x = torch.randn(1000, 1000)  # Default CPU placement
print("Where is this tensor sitting?:", x.device)

### Step 2: Routing the Tensor to GPU

To fix this, we must explicitly declare our target device and copy the tensor over.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
x_gpu = x.to(device)  # Move data to GPU
print("Where is the tensor sitting now?:", x_gpu.device)

### Interpretation

Now the tensor resides on `cuda:0`. Any subsequent mathematical operations on `x_gpu` will be executed in parallel by the GPU's CUDA cores instead of the CPU. If you encounter 0% GPU utilization, checking the `.device` attribute is your first debugging step!

### Module 1 Recap

* Enabled GPU hardware is wasted if PyTorch is not instructed to use it.
* Use **`tensor.device`** to verify where data is actually sitting.
* Always route your variables using **`.to("cuda")`** to utilize the GPU.